This script takes a segmentation and spots to calculate the number of spots / cell.

## Input:
- segmentation masks in either `npy` or `png` format (name should correspond to the name of the image in the spot file)
- `.csv` file of spots containing:
    - *img* - image name
    - *channel* - channel number the spot belongs to  
    - *x, y* and *z* coordiantes

## Output
- `spots_per_cell.csv` with information on how many spots each cell in an image and channel contains.

# Functions and imports
*also part of `pipelines/fish_utils`*

In [ ]:
import os
from glob import glob
import pandas as pd
import numpy as np
import re

from skimage.morphology import label
from skimage.segmentation import clear_border
from skimage.measure import regionprops

In [345]:
def get_sensitivity(masks,path_spots,out,mask_ending="_cp_masks"):
    
    df = pd.read_csv(path_spots)
        
    df_list = []
    
    for file in masks:
        # subset spots for spots in current image
        name = file.split("/")[-1].split(".")[0]
        name = name.replace(mask_ending, "")
        name = re.sub(r'_ch\d+', '', name)
        subset_df = df[df['img'].str.contains(name)]
        
        # load mask
        file_type = os.path.splitext(file)[1]
        
        if file_type == ".npy":
            mask = np.load(file,allow_pickle=True).item()['masks']
        elif file_type == ".png":
            mask = imread(file)
        else:
            print("Please input valid segmentation masks (.npy and .png supported).")
        
        # label all cells and remove cells on edges
        labelled_mask = label(mask)
#         cleared_mask = clear_border(labelled_mask)
        cleared_mask = labelled_mask

        
        # add cell info to spots
        # for 2d masks
        if len(cleared_mask.shape) == 2:            
            cell = labelled_mask[subset_df['y'].astype(int), subset_df['x'].astype(int)]
            subset_df.loc[:, 'cell'] = cell if 'cell' in subset_df.columns else cell
#             subset_df.insert(1, 'cell', cell)
            
        # for 3d masks
        elif len(cleared_mask.shape) == 3:
            cell = labelled_mask[subset_df['z'].astype(int), subset_df['y'].astype(int), subset_df['x'].astype(int)]
            subset_df.loc[:, 'cell'] = cell if 'cell' in subset_df.columns else cell
#             subset_df.insert(1, 'cell', cell)
    

        # add number of spots in each cell
        img_names = pd.DataFrame({'img': subset_df['img'].unique()}) # all unique img names
        cell_df = pd.DataFrame({'cell': np.unique(cleared_mask)}) # all cells
        cell_df = cell_df.merge(img_names,how='cross')
        spots = subset_df.groupby(['img','cell']).size().reset_index(name='count') # cell id for each spot        
        cell_df = spots.merge(cell_df, on=['img','cell'], how='outer')
        cell_df['count'].fillna(0, inplace=True) # all cells without spots get 0
        
        # measure cell sizes using regionprops
        region_props = regionprops(cleared_mask)
        cell_sizes = [[prop.label, prop.area] for prop in region_props]
        cell_sizes = pd.DataFrame(cell_sizes, columns=['cell', 'cell_size'])
        cell_df = cell_df.merge(cell_sizes, on='cell', how='outer')
        
        df_list.append(cell_df)
      
    # combine all images into 1 df
    spots_per_cell = pd.concat(df_list, ignore_index=True) 
    
    # remove too small cells (faulty segmentation)
    spots_per_cell = spots_per_cell[spots_per_cell['cell_size'] > 50000]
    
#     # get sensitivity
#     spots_per_cell = spots_per_cell[spots_per_cell.cell != 0]

    # add metadata
    columns_to_drop = ['x', 'y', 'z', 'spot_idx', 't', 'c', 'intensity','cell','whole_cell']
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns]) # crop spot specific cols
    
    spots_per_cell = spots_per_cell.merge(df,on="img",how="left")
    spots_per_cell = spots_per_cell.dropna(subset=["channel"]).drop_duplicates()
    
    spots_per_cell.to_csv(out, index=False)

# Get number of spots per cell

In [159]:
in_path =  None #upper level experiment folder
mask_ending = "_seg"
rel_spot_path = "/detections/merge_filtered.csv"

In [346]:
path_spots = f"{in_path}/{rel_spot_path}" #spots file
masks = glob(f"{in_path}/segmentation/*.npy") #all segmentation masks (.npy and .png supported)
out = f"{in_path}/detections/spots_per_cell.csv" # where to save new file

get_sensitivity(masks,path_spots,out,mask_ending=mask_ending)